# Dark-majority evidence review

**Status:** completed exploratory companion notebook  
**Research thread:** unresolved commercialization outcomes and dark-firm follow-up  
**Canonical computation:** WS5/WS6 scripts and `nano_capture_recapture.py` under `scripts/data/`

Use this notebook as the template for investigations where missing evidence, search coverage, and true negative evidence must remain separate.

In [ ]:
from pathlib import Path

import pandas as pd


def find_repo_root(start: Path = Path.cwd()) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "sbir_etl").exists():
            return candidate
    raise RuntimeError("Run this notebook from inside the sbir-analytics checkout")


REPO_ROOT = find_repo_root()
AREA_ID = "nanotechnology"
REPORT_DIR = REPO_ROOT / "data" / "reports" / AREA_ID
RANDOM_SEED = 20260804

## Data contract

The dark population was selected for absence of primary transition signals. Secondary channels answer narrower questions about observable activity. `not found`, `not searched`, `API unavailable`, and affirmative negative evidence are different states and must not be collapsed.

In [ ]:
ARTIFACTS = {
    "liveness": REPORT_DIR / "dark_firm_liveness.csv",
    "trademarks": REPORT_DIR / "dark_firm_trademarks.csv",
    "alias evidence": REPORT_DIR / "alias_expanded_evidence.csv",
    "subawards": REPORT_DIR / "ws5a_subawards.csv",
    "sector registries": REPORT_DIR / "ws5c_sector_registries.csv",
    "dark capture matrix": REPORT_DIR / "capture_recapture_darkfirms.csv",
}
availability = pd.DataFrame(
    [{"artifact": name, "path": path.relative_to(REPO_ROOT), "exists": path.exists()} for name, path in ARTIFACTS.items()]
)
availability

## Secondary-channel coverage

Use the canonical dark-population capture matrix for common-population overlap. Do not add sector-registry evidence to that matrix when only a sector subset was searched.

In [ ]:
capture_path = ARTIFACTS["dark capture matrix"]
if not capture_path.exists():
    print(
        f"Missing {capture_path.relative_to(REPO_ROOT)}. Generate canonical WS artifacts "
        f"before running nano_capture_recapture.py --area {AREA_ID}."
    )
    capture = pd.DataFrame()
    overlap = pd.DataFrame()
else:
    capture = pd.read_csv(capture_path)
    channels = [column for column in ["patent", "trademark", "alias", "subaward"] if column in capture]
    binary = capture[channels].fillna(0).astype(int)
    overlap = binary.T.dot(binary)
overlap

In [ ]:
if capture.empty:
    unresolved_sample = pd.DataFrame()
else:
    unresolved = capture[capture["n_channels"].eq(0)]
    unresolved_sample = unresolved.sample(min(25, len(unresolved)), random_state=RANDOM_SEED)
unresolved_sample

## Coverage and provenance audit

Before interpreting unresolved firms, record whether every firm was eligible for and actually searched by each source.

In [ ]:
artifact_shapes = {}
for name, path in ARTIFACTS.items():
    if path.exists():
        frame = pd.read_csv(path, low_memory=False)
        artifact_shapes[name] = {"rows": len(frame), "columns": len(frame.columns)}
pd.DataFrame.from_dict(artifact_shapes, orient="index")

## Review log

| Firm/sample | Source eligibility | Search completed? | Positive/negative/unknown | Follow-up |
|---|---|---|---|---|
| _Draft_ | _State sector/source scope_ | _Yes/no_ | _Do not collapse states_ | _Next instrument_ |

Any revised published count must be regenerated through the canonical scripts and checked by `nano_verify_report_figures.py`.